# Blog 1 Notebook 01: Explore The SQL-Agent Benchmark

This notebook owns benchmark exploration and split creation for Blog 1. The full eval and training runs stay in Python scripts. Here we inspect the raw dataset, understand the harness, choose a fixed train/eval split, and write the prepared data files used by later scripts.

![SQL tool-use harness](../assets/sql-tool-use-harness-sketchnote.png)

The model sees the user issue, buggy SQL, and environment observations. It does not see the hidden tests or reference SQL during inference.

In [1]:
import json
import random
import sys
from collections import Counter
from pathlib import Path

from datasets import load_dataset

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "common" / "sql_agent.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common import config as cfg
from common import generation
from common import sql_agent

DATA_DIR = PROJECT_ROOT / "data" / "sql_agent_bird_critic"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

print("Project root:", PROJECT_ROOT)
print("Dataset:", cfg.SQL_AGENT_DATASET)
print("Prepared data dir:", DATA_DIR)

Project root: /home/isaac/codes/personal/small-lm-training
Dataset: birdsql/six-gym-sqlite
Prepared data dir: /home/isaac/codes/personal/small-lm-training/data/sql_agent_bird_critic


## Load And Inspect The Raw Dataset

This is the source benchmark. We normalize rows into the compact schema used by the harness, then inspect distributions before writing any split.

In [2]:
raw_dataset = load_dataset(cfg.SQL_AGENT_DATASET, split="train")
raw_dataset

Dataset({
    features: ['instance_id', 'issue_sql', 'dialect', 'version', 'db_id', 'clean_up_sql', 'test_cases', 'sol_sql', 'query', 'preprocess_sql', 'category'],
    num_rows: 5000
})

In [3]:
rows = [sql_agent.normalize_source_row(row) for row in raw_dataset]

In [4]:
rows[0]

{'id': 'TRAIN_0',
 'instance_id': 'TRAIN_0',
 'db_id': 'movie_3',
 'category': 'Query',
 'query': "I'm working with customer rental data and trying to flag rentals based on their timing relative to previous ones for the same customer. My goal is to mark a rental with a '1' if its `rental_date` occurs within 90 days of the `return_date` of the most recent *unflagged* rental for that customer. The very first rental for any customer should always be unflagged (marked '0').\n\nThe tricky part is that once a rental is flagged ('1'), it shouldn't be used as the starting point for the 90-day check for subsequent rentals. Only rentals marked '0' should reset this reference point. For instance, if a customer has rentals starting on May 24th (unflagged '0'), June 15th (flagged '1' because it's within 90 days of May 24th's return date), and August 20th, the August 20th rental should be compared back to the May 24th rental (the last unflagged one). If August 20th is more than 90 days after the *re

### What Each Dataset Field Means

Each row is one executable SQL-agent task. It contains the user-facing problem, the buggy SQL attempt, and the hidden setup/scoring information used by the benchmark.

- `id` / `instance_id`: unique task identifier. We use this for train/eval splits, result caching, and trace lookup.

- `db_id`: the SQLite database template for this task. The harness copies this database into a temporary task database before running the model.

- `category`: the task category from the dataset, such as `Query`, `Management`, or `Personalization`.

- `query`: the natural-language user issue. This is the main task description the model sees.

- `issue_sql`: the buggy or incomplete SQL supplied by the user. The model sees this and should repair or replace it.

- `sol_sql`: the reference solution SQL. The model does **not** see this during inference. The evaluator uses it to compare against the model’s submitted SQL.

- `preprocess_sql`: setup SQL executed before evaluation. This may create temp tables, insert test data, or prepare the database state.

- `clean_up_sql`: cleanup SQL from the dataset, usually to drop temporary objects. In our harness each task uses a temporary copied database, so cleanup is mostly informational.

- `test_cases`: hidden Python scoring code. The evaluator runs these tests after the model submits SQL. Most tests compare the model’s SQL result with the reference `sol_sql`.

So the model sees roughly:

```text
db_id + category + query + issue_sql + tool observations
```

The evaluator uses:

```text
preprocess_sql + submitted SQL + sol_sql + test_cases
```

That is why this is more than a static text-to-SQL row: it is an executable task. The model can inspect schema, run SQL, observe results/errors, and then submit final SQL for deterministic scoring.


In [5]:
print("Raw rows:", len(rows))
print("Columns in one normalized row:")
print(list(rows[0].keys()))
print("Categories:")
for category, count in Counter(row["category"] for row in rows).most_common():
    print(f"  {category}: {count}")
print("Databases:", len({row["db_id"] for row in rows}))

Raw rows: 5000
Columns in one normalized row:
['id', 'instance_id', 'db_id', 'category', 'query', 'issue_sql', 'sol_sql', 'preprocess_sql', 'clean_up_sql', 'test_cases']
Categories:
  Query: 3094
  Management: 1087
  Personalization: 819
Databases: 13


In [6]:
issue_sql_counts = Counter(len(row["issue_sql"]) for row in rows)
test_case_counts = Counter(len(row["test_cases"]) for row in rows)
category_db_counts = Counter((row["category"], row["db_id"]) for row in rows)

print("Issue SQL statement count distribution:", dict(sorted(issue_sql_counts.items())))
print("Test case count distribution:", dict(sorted(test_case_counts.items())))
print("Most common category/database pairs:")
for (category, db_id), count in category_db_counts.most_common(12):
    print(f"  {category:24s} {db_id:35s} {count}")

Issue SQL statement count distribution: {1: 4546, 2: 282, 3: 78, 4: 44, 5: 18, 6: 18, 7: 5, 8: 4, 10: 1, 11: 1, 14: 2, 25: 1}
Test case count distribution: {1: 4971, 2: 23, 3: 4, 5: 2}
Most common category/database pairs:
  Query                    olympics                            311
  Query                    netflix                             293
  Query                    lego                                282
  Query                    books                               282
  Query                    movie_3                             273
  Query                    book_publishing_company             271
  Query                    employees                           270
  Query                    chinook                             251
  Query                    address                             226
  Query                    airline                             218
  Query                    public_review_platform              162
  Management               books         

## Choose The Split

For the blog we first narrow the dataset to one task category and a small group of related databases. Then we split by percentage inside each database, so train and eval keep the same domain mix.

In [7]:
SPLIT_SEED = 42
EVAL_FRACTION = cfg.SQL_AGENT_EVAL_FRACTION
TASK_CATEGORY = "Query"
DB_FILTER = ["netflix", "movie_3", "books", "chinook"]

In [8]:
candidate_rows = [row for row in rows if row["category"] == TASK_CATEGORY and row["db_id"] in DB_FILTER]
candidate_counts = Counter(row["db_id"] for row in candidate_rows)

print("Filtered candidate rows:", len(candidate_rows))
print("Candidate rows by database:")
for db_id in DB_FILTER:
    print(f"  {db_id:12s} {candidate_counts[db_id]}")

Filtered candidate rows: 1099
Candidate rows by database:
  netflix      293
  movie_3      273
  books        282
  chinook      251


In [9]:
assert 0 < EVAL_FRACTION < 1

rng = random.Random(SPLIT_SEED)
train_rows = []
eval_rows = []

for db_id in DB_FILTER:
    db_rows = [row for row in candidate_rows if row["db_id"] == db_id]
    rng.shuffle(db_rows)
    eval_count = max(1, round(len(db_rows) * EVAL_FRACTION))
    eval_rows.extend(db_rows[:eval_count])
    train_rows.extend(db_rows[eval_count:])

rng.shuffle(train_rows)
rng.shuffle(eval_rows)
selected_rows = train_rows + eval_rows

stats = {
    "dataset": cfg.SQL_AGENT_DATASET,
    "seed": SPLIT_SEED,
    "task_category": TASK_CATEGORY,
    "db_filter": DB_FILTER,
    "eval_fraction": EVAL_FRACTION,
    "source_size": len(rows),
    "candidate_size": len(candidate_rows),
    "train_size": len(train_rows),
    "eval_size": len(eval_rows),
    "db_count": len({row["db_id"] for row in selected_rows}),
    "candidate_db_counts": dict(sorted(candidate_counts.items())),
    "train_db_counts": dict(sorted(Counter(row["db_id"] for row in train_rows).items())),
    "eval_db_counts": dict(sorted(Counter(row["db_id"] for row in eval_rows).items())),
    "categories": sql_agent.category_counts(selected_rows),
}
print(json.dumps(stats, indent=2))

{
  "dataset": "birdsql/six-gym-sqlite",
  "seed": 42,
  "task_category": "Query",
  "db_filter": [
    "netflix",
    "movie_3",
    "books",
    "chinook"
  ],
  "eval_fraction": 0.2,
  "source_size": 5000,
  "candidate_size": 1099,
  "train_size": 879,
  "eval_size": 220,
  "db_count": 4,
  "candidate_db_counts": {
    "books": 282,
    "chinook": 251,
    "movie_3": 273,
    "netflix": 293
  },
  "train_db_counts": {
    "books": 226,
    "chinook": 201,
    "movie_3": 218,
    "netflix": 234
  },
  "eval_db_counts": {
    "books": 56,
    "chinook": 50,
    "movie_3": 55,
    "netflix": 59
  },
  "categories": {
    "Query": 1099
  }
}


In [10]:
print("Train rows:", len(train_rows))
print("Eval rows:", len(eval_rows))
print("Actual eval fraction:", round(len(eval_rows) / len(selected_rows), 4))

Train rows: 879
Eval rows: 220
Actual eval fraction: 0.2002


## Write The Prepared Split

This replaces the old dataset-preparation script. Running this cell writes the files consumed by eval and teacher-generation scripts. The database templates are large, so they are downloaded/copied into `data/sql_agent_bird_critic/dbs/` and ignored by git.

In [11]:
WRITE_PREPARED_SPLIT = True

if WRITE_PREPARED_SPLIT:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    cfg.write_jsonl(DATA_DIR / "train.jsonl", train_rows)
    cfg.write_jsonl(DATA_DIR / "eval.jsonl", eval_rows)
    cfg.write_json(DATA_DIR / "stats.json", stats)
    sql_agent.copy_database_templates(DATA_DIR, {row["db_id"] for row in selected_rows})
    print("Wrote:", DATA_DIR / "train.jsonl")
    print("Wrote:", DATA_DIR / "eval.jsonl")
    print("Wrote:", DATA_DIR / "stats.json")
    print("Database templates:", DATA_DIR / "dbs")
else:
    print("Set WRITE_PREPARED_SPLIT = True when you want to write the split.")

Wrote: /home/isaac/codes/personal/small-lm-training/data/sql_agent_bird_critic/train.jsonl
Wrote: /home/isaac/codes/personal/small-lm-training/data/sql_agent_bird_critic/eval.jsonl
Wrote: /home/isaac/codes/personal/small-lm-training/data/sql_agent_bird_critic/stats.json
Database templates: /home/isaac/codes/personal/small-lm-training/data/sql_agent_bird_critic/dbs


## Inspect A Real Eval Task

The reference SQL and tests are shown for us as readers. The model will not see them during inference.

In [12]:
TASK_ID = eval_rows[0]["id"]
row_by_id = {row["id"]: row for row in selected_rows}
row = row_by_id[TASK_ID]

print("Task id:", row["id"])
print("Database:", row["db_id"])
print("Category:", row["category"])
print("\nUser issue:\n", row["query"])
print("\nBuggy SQL:")
for sql in row["issue_sql"]:
    print(sql)
print("\nReference SQL, hidden from the model during inference:")
for sql in row["sol_sql"]:
    print(sql)
print("\nNumber of hidden tests:", len(row["test_cases"]))

Task id: TRAIN_175
Database: movie_3
Category: Query

User issue:
 We have a simulated payment history table (`payment_fifo_demo`) that records hypothetical buying and selling of films. Each record has a unique payment ID, the film ID, the timestamp of the transaction, whether it was a buy or sell (`is_sell`), the number of film units (`quantity`), and the total amount (`total_amount`) for the transaction. For reporting purposes (similar to inventory cost allocation), we need to match each sell transaction with the corresponding buy transactions, allocating the units sold from the earliest buy transactions first (FIFO). The result should include the film ID, the buy timestamp, the sell timestamp, the number of units matched in the transaction, the total cost allocated from the buy transactions, and the total revenue allocated from the sell transactions for those matched units. For example, if we have these records in `payment_fifo_demo`:

|payment_id | film_id | payment_timestamp  | is

## What Goes Into The Model

The first call contains the system action contract and the user task. After each tool action, the harness appends an environment observation and asks for the next JSON action.

In [13]:
row

{'id': 'TRAIN_175',
 'instance_id': 'TRAIN_175',
 'db_id': 'movie_3',
 'category': 'Query',
 'query': "We have a simulated payment history table (`payment_fifo_demo`) that records hypothetical buying and selling of films. Each record has a unique payment ID, the film ID, the timestamp of the transaction, whether it was a buy or sell (`is_sell`), the number of film units (`quantity`), and the total amount (`total_amount`) for the transaction. For reporting purposes (similar to inventory cost allocation), we need to match each sell transaction with the corresponding buy transactions, allocating the units sold from the earliest buy transactions first (FIFO). The result should include the film ID, the buy timestamp, the sell timestamp, the number of units matched in the transaction, the total cost allocated from the buy transactions, and the total revenue allocated from the sell transactions for those matched units. For example, if we have these records in `payment_fifo_demo`:\n\n|payment_

In [14]:
messages = sql_agent.initial_messages(row)
for message in messages:
    print("=" * 80)
    print(message["role"].upper()+":")
    print(message["content"][:3000])

SYSTEM:
You are a SQL tool-use agent.
You repair or write SQLite for a user issue by interacting with a deterministic database environment.

Return exactly one structured JSON decision per assistant turn. Do not write prose outside JSON.

Decision shape:
{"draft": "short plan, max 18 words", "output": {"action": "inspect_schema"}}
{"draft": "short plan, max 18 words", "output": {"action": "run_sql_query", "sql": "SELECT ..."}}
{"draft": "short plan, max 18 words", "output": {"action": "submit_sql", "sql": ["SQL statement 1", "SQL statement 2"]}}

Available actions:
{"action": "inspect_schema"}
{"action": "run_sql_query", "sql": "SELECT ..."}
{"action": "submit_sql", "sql": ["SQL statement 1", "SQL statement 2"]}

Rules:
- Use inspect_schema before writing SQL if the schema is not obvious.
- Use run_sql_query to test SQL and inspect errors or result rows.
- Use submit_sql only when you are ready to submit the final corrected SQL.
- The submitted SQL must be SQLite.
- If the final answer

## The Action Contract

The model must return exactly one JSON action object per assistant turn.

In [15]:
print(sql_agent.SYSTEM_PROMPT)

You are a SQL tool-use agent.
You repair or write SQLite for a user issue by interacting with a deterministic database environment.

Return exactly one structured JSON decision per assistant turn. Do not write prose outside JSON.

Decision shape:
{"draft": "short plan, max 18 words", "output": {"action": "inspect_schema"}}
{"draft": "short plan, max 18 words", "output": {"action": "run_sql_query", "sql": "SELECT ..."}}
{"draft": "short plan, max 18 words", "output": {"action": "submit_sql", "sql": ["SQL statement 1", "SQL statement 2"]}}

Available actions:
{"action": "inspect_schema"}
{"action": "run_sql_query", "sql": "SELECT ..."}
{"action": "submit_sql", "sql": ["SQL statement 1", "SQL statement 2"]}

Rules:
- Use inspect_schema before writing SQL if the schema is not obvious.
- Use run_sql_query to test SQL and inspect errors or result rows.
- Use submit_sql only when you are ready to submit the final corrected SQL.
- The submitted SQL must be SQLite.
- If the final answer needs m

## One Manual Harness Step

This cell shows what the SQLite environment returns after an action. We use `inspect_schema` because it is safe and does not need a model.

In [16]:
template = sql_agent.template_path(DATA_DIR, row["db_id"])
if not template.exists():
    print("Database template is not prepared yet:", template)
    print("Run the write-split cell below first, then rerun this cell.")
else:
    with sql_agent.task_database(row, DATA_DIR) as db_path:
        import sqlite3
        conn = sqlite3.connect(db_path)
        try:
            sql_agent.execute_sql_list(conn, row["preprocess_sql"])
            print(sql_agent.schema_text(conn)[:4000])
        finally:
            conn.close()

-- main table
CREATE TABLE "actor"
(
    actor_id    INTEGER
        primary key autoincrement,
    first_name  TEXT                               not null,
    last_name   TEXT                               not null,
    last_update DATETIME default CURRENT_TIMESTAMP not null
)

columns: actor_id INTEGER, first_name TEXT, last_name TEXT, last_update DATETIME

-- main table
CREATE TABLE "address"
(
    address_id  INTEGER
        primary key autoincrement,
    address     TEXT                               not null,
    address2    TEXT,
    district    TEXT                               not null,
    city_id     INTEGER                            not null
        references city
            on update cascade,
    postal_code TEXT,
    phone       TEXT                               not null,
    last_update DATETIME default CURRENT_TIMESTAMP not null
)

columns: address_id INTEGER, address TEXT, address2 TEXT, district TEXT, city_id INTEGER, postal_code TEXT, phone TEXT, last_update DA

## Optional Live One-Task Run

The harness calls an OpenAI-compatible HTTP server through BAML. On this Mac, start the MLX server for the base student first:

```bash
uv run mlx_lm.server \
  --model mlx-community/Qwen3.5-0.8B-MLX-bf16 \
  --host 127.0.0.1 \
  --port 8091 \
  --chat-template-args '{"enable_thinking": false}'
```

On NVIDIA later, serve the same student family through vLLM:

```bash
vllm serve Qwen/Qwen3.5-0.8B \
  --host 127.0.0.1 \
  --port 8091 \
  --served-model-name Qwen/Qwen3.5-0.8B \
  --max-model-len 32768 \
  --gpu-memory-utilization 0.75 \
  --default-chat-template-kwargs '{"enable_thinking": false}'
```

The model name sent to BAML must match the server's `/v1/models` id. For the Mac command above, use `cfg.MLX_STUDENT_MODEL`. For the NVIDIA command, use `cfg.HF_STUDENT_MODEL`.


In [17]:
LIVE_STUDENT_BASE_URL = "http://127.0.0.1:8091/v1"
LIVE_STUDENT_MODEL = cfg.MLX_STUDENT_MODEL

required_template = sql_agent.template_path(DATA_DIR, row["db_id"])
if not required_template.exists():
    print("Missing database template for the live row. Writing the prepared split first.")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    cfg.write_jsonl(DATA_DIR / "train.jsonl", train_rows)
    cfg.write_jsonl(DATA_DIR / "eval.jsonl", eval_rows)
    cfg.write_json(DATA_DIR / "stats.json", stats)
    sql_agent.copy_database_templates(DATA_DIR, {row["db_id"] for row in selected_rows})

print("Live model:", LIVE_STUDENT_MODEL)
print("Live base URL:", LIVE_STUDENT_BASE_URL)

import httpx
try:
    models_response = httpx.get(f"{LIVE_STUDENT_BASE_URL}/models", timeout=5)
    models_response.raise_for_status()
except Exception as exc:
    raise RuntimeError(
        "No OpenAI-compatible model server is reachable yet. Start the vLLM or MLX command above, wait until /v1/models responds, then rerun this cell."
    ) from exc

generate = generation.make_baml_generator(
    model_name=LIVE_STUDENT_MODEL,
    base_url=LIVE_STUDENT_BASE_URL,
    max_new_tokens=4096,
)
live_result = sql_agent.run_task(row, data_dir=DATA_DIR, generate=generate, max_turns=8, keep_messages=True)
print(json.dumps(cfg.make_json_safe(live_result), indent=2, ensure_ascii=False))


Live model: Qwen/Qwen3.5-0.8B
Live base URL: http://127.0.0.1:8091/v1
{
  "id": "TRAIN_175",
  "db_id": "movie_3",
  "category": "Query",
  "success": false,
  "stop_reason": "max_turns",
  "turns": 8,
  "trace": [
    {
      "turn": 1,
      "baml_output": "{\"draft\": \"Check if the FIFO logic is correctly implemented in the current SQL query.\", \"output\": {\"action\": \"run_sql_query\", \"sql\": \"SELECT film_id, payment_timestamp AS buy_timestamp, payment_timestamp AS sell_timestamp, quantity, total_amount AS buy_total_amount, total_amount AS sell_total_amount FROM payment_fifo_demo WHERE is_sell = 0 AND is_sell = 1 ORDER BY payment_timestamp, payment_timestamp;\"}}",
      "draft": "Check if the FIFO logic is correctly implemented in the current SQL query.",
      "action": {
        "action": "run_sql_query",
        "sql": "SELECT film_id, payment_timestamp AS buy_timestamp, payment_timestamp AS sell_timestamp, quantity, total_amount AS buy_total_amount, total_amount AS sell_

In [18]:
for turn in live_result['trace']:
    print(turn['baml_output'])
    print()

{"draft": "Check if the FIFO logic is correctly implemented in the current SQL query.", "output": {"action": "run_sql_query", "sql": "SELECT film_id, payment_timestamp AS buy_timestamp, payment_timestamp AS sell_timestamp, quantity, total_amount AS buy_total_amount, total_amount AS sell_total_amount FROM payment_fifo_demo WHERE is_sell = 0 AND is_sell = 1 ORDER BY payment_timestamp, payment_timestamp;"}}

{"draft": "The current query returns no rows, indicating the FIFO logic is not implemented or the join condition is incorrect.", "output": {"action": "run_sql_query", "sql": "SELECT film_id, payment_timestamp AS buy_timestamp, payment_timestamp AS sell_timestamp, quantity, total_amount AS buy_total_amount, total_amount AS sell_total_amount FROM payment_fifo_demo WHERE is_sell = 0 AND is_sell = 1 ORDER BY payment_timestamp, payment_timestamp;"}}

{"draft": "The current query returns no rows, indicating the FIFO logic is not implemented or the join condition is incorrect.", "output": {"

## Optional Saved Eval Traces

After running the eval scripts, this cell summarizes saved results and lets you inspect real traces without rerunning the benchmark inside the notebook.

In [19]:
result_paths = {
    "base_student": OUTPUT_DIR / "qwen3_5_0_8b_mlx_sql_agent_eval.json",
    "gpt_teacher": OUTPUT_DIR / "gpt_5_5_medium_sql_agent_eval.json",
    "qwen35_teacher": OUTPUT_DIR / "qwen3_5_35b_a3b_8bit_mlx_server_sql_agent_eval.json",
}

loaded_results = {}
for name, path in result_paths.items():
    if not path.exists():
        print(f"{name}: missing {path}")
        continue
    data = json.loads(path.read_text())
    loaded_results[name] = {item["id"]: item for item in data["results"]}
    summary = data["summary"]
    print(f"{name}: success={summary['success']}/{summary['total']} submitted={summary['submitted']} parse_failures={summary['parse_failures']}")

base_student: missing /home/isaac/codes/personal/small-lm-training/outputs/qwen3_5_0_8b_mlx_sql_agent_eval.json
gpt_teacher: missing /home/isaac/codes/personal/small-lm-training/outputs/gpt_5_5_medium_sql_agent_eval.json
qwen35_teacher: missing /home/isaac/codes/personal/small-lm-training/outputs/qwen3_5_35b_a3b_8bit_mlx_server_sql_agent_eval.json
